#  Hexar AI Model

### fine-tuning of qwen3-8B

**Gaming Dataset**

In [ ]:
# Install necessary libraries for fine-tuning
!pip install -U transformers accelerate peft bitsandbytes trl datasets

Now, let's load the dataset you specified for fine-tuning.

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset_name = "markov-ai/gaming-500-hours"
dataset = load_dataset(dataset_name)

print(f"Dataset loaded: {dataset_name}")
print(dataset)

Next, we'll load the base model and tokenizer, applying 4-bit quantization to optimize memory usage, which is crucial for fine-tuning large models like Qwen3-8B on a T4 GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

# Load the base model with quantization
model_id = "Qwen/Qwen3-8B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True # Qwen models might require this
)
model.config.use_cache = False # Disable cache for fine-tuning
model.config.pretraining_tp = 1 # Required for some models

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token # Set pad token
tokenizer.padding_side = "right" # Pad to the right

Now, we prepare the model for Parameter-Efficient Fine-Tuning (PEFT) using the LoRA method. This allows us to train only a small number of additional parameters, making the process much faster and less memory-intensive than full fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model

# LoRA configuration
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

# The SFTTrainer will apply the PEFT model itself when peft_config is provided.

Next, we'll define the training arguments and use the `SFTTrainer` for supervised fine-tuning. The `SFTTrainer` simplifies the process of training language models on instruction-following datasets.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

# Define a formatting function to combine relevant fields into a 'text' column
def formatting_func(example):
    # Combine title and description into a conversational format
    output_text = f"### System:\nYou are Hexar, a helpful AI assistant focused on gaming.\n### Instruction:\n{example['title']}\n### Response:\n{example['description']}"
    # Return a new dictionary with only the 'text' field
    return {"text": output_text}

# Apply the formatting function to the training dataset before passing it to SFTTrainer
formatted_train_dataset = dataset['train'].map(formatting_func, remove_columns=dataset['train'].column_names)

# Training arguments
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1, # You can adjust this
    per_device_train_batch_size=2, # Adjust based on GPU memory
    gradient_accumulation_steps=1, # Adjust based on GPU memory
    optim="paged_adamw_32bit",
    save_steps=100, # Save checkpoint every X steps
    logging_steps=10, # Log metrics every X steps
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=not torch.cuda.is_bf16_supported(), # Use fp16 if bfloat16 is not supported
    bf16=torch.cuda.is_bf16_supported(), # Use bf16 if supported
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=12, # Replaced warmup_ratio with warmup_steps (0.03 * 776 / 2 = 11.64, rounded up)
    lr_scheduler_type="constant",
    report_to="none" # Can be "wandb", "tensorboard", etc. if you set them up
)

# Initialize SFTTrainer with the pre-formatted dataset
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset, # Use the pre-formatted dataset
    peft_config=peft_config,
    args=training_arguments,
    # formatting_func=formatting_func # Removed as dataset is already formatted
)

Finally, let's start the fine-tuning process. This might take some time depending on your dataset size and chosen training parameters.

In [ ]:
# Start training
trainer.train()

# Save the fine-tuned model
trainer.save_model("qwen3-8b-Hexar-Gaming-Model")
print("Fine-tuning complete! Model saved to qwen3-8b-Hexar-Gaming-Model.")

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, message="You are using a model that has been trained with a different tokenizer")

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("qwen3-8b-Hexar-Gaming-Model")
model = AutoModelForCausalLM.from_pretrained("qwen3-8b-Hexar-Gaming-Model")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! You are using a GPU.")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is NOT available. You might be running on a CPU or need to change your Colab runtime type to GPU.")
    print("To change the runtime type: Go to 'Runtime' -> 'Change runtime type' -> Select 'T4 GPU' or 'V100 GPU' (if available) under 'Hardware accelerator', then click 'Save'.")